# Notebook 02 — Replication of RQ1 (Wu et al., EMSE 2023)

Notebook port of the team pipeline in `src/` — same algorithms, parameters and seeds, adapted
to run on Google Colab (Drive caching so the expensive parsing step runs only once).

| Step | Script ported | What it does |
|---|---|---|
| 1 | `src/parse.py` | Drain (drain3) + regex preprocessing → event templates |
| 2 | `src/partition.py`, `src/split.py` | sessions (HDFS = block_id, BGL = 6 h window), stratified split, seed 0 |
| 3 | `src/mcv.py`, `src/tfidf_text.py` | MCV features (for RF) + TF-IDF (Text) event vectors (for LSTM) |
| 4 | `src/run_rf.py`, `src/run_lstm.py`, `src/feature_selection.py` | RF + LSTM, each with and without correlation+VIF selection |
| 5 | `src/summarize_rq1.py` | metrics vs the paper |

Model ↔ representation (one representation per model, as in the team's scripts):

- **Random Forest** (classical) on **MCV** — message count vector, sequence-level
- **LSTM** (deep learning) on **TF-IDF (Text)** — event-level vectors, sliding windows
  (HDFS 30/1, BGL 50/50), session score = max anomaly probability over its windows

Reference F1 (paper Tables 2–3 for the corresponding representation, and the team's
`results/rq1_summary.csv` run with seed 0):

| Model | Dataset | Paper | Team (seed 0, this notebook should be close) |
|---|---|---|---|
| RF (MCV) | HDFS | 0.999 | 0.9974 |
| RF (MCV) | BGL | 0.891 | 0.9317 |
| LSTM (TF-IDF Text) | HDFS | 0.955 | 0.9409 |
| LSTM (TF-IDF Text) | BGL | 0.871 | 0.9061 |

Runtime: HDFS parsing ≈ 20–40 min (cached to Drive afterwards); RF ≤ 5 min; LSTM ≤ 10 min
per run. `Runtime → Change runtime type → GPU` speeds the LSTM up but is not required.

In [ ]:
%pip install -q drain3 tqdm

In [ ]:
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time

SEED = 0   # seed 0 = main replication split (seeds 1-9 were used for the Scott-Knott extension)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    REPO = '/content/DevOps'
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', 'https://github.com/TranHap/DevOps.git', REPO], check=True)
    DRIVE_ROOT = '/content/drive/MyDrive/LOG6309E'   # populated by notebook 01
else:
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'src')):
        REPO = os.path.dirname(REPO)
    DRIVE_ROOT = os.path.join(REPO, 'notebooks', 'cache')   # local: data/ must already exist

CACHE = os.path.join(DRIVE_ROOT, 'cache')   # parsed/sessions artifacts survive Colab resets
os.makedirs(CACHE, exist_ok=True)
print('REPO      =', REPO)
print('DRIVE_ROOT=', DRIVE_ROOT)

RAW = {'HDFS': ['HDFS/HDFS.log', 'HDFS/preprocessed/anomaly_label.csv'], 'BGL': ['BGL/BGL.log']}

def stage(ds):
    # 1) parsed artifacts cached on Drive by a previous run? -> no re-parsing needed
    pdir = os.path.join(REPO, 'data', 'parsed', ds)
    cdir = os.path.join(CACHE, 'parsed', ds)
    if not os.path.exists(os.path.join(pdir, ds + '_events.csv')) \
            and os.path.exists(os.path.join(cdir, ds + '_events.csv')):
        os.makedirs(pdir, exist_ok=True)
        for f in glob.glob(os.path.join(cdir, ds + '_*.csv')):
            shutil.copy2(f, pdir)
        print(f'[cache] restored parsed {ds} from Drive')
    # 2) raw log (+ labels) from Drive into the repo layout
    for rel in RAW[ds]:
        dest = os.path.join(REPO, 'data', 'raw', rel)
        if not os.path.exists(dest):
            src = os.path.join(DRIVE_ROOT, 'data', rel)
            assert os.path.exists(src), f'missing {src} - run notebook 01 first'
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            print(f'copying {rel} ({os.path.getsize(src) / 1e6:.0f} MB) ...', flush=True)
            shutil.copy2(src, dest)

for ds in ('HDFS', 'BGL'):
    stage(ds)
print('data ready under', os.path.join(REPO, 'data', 'raw'))

## Step 1 — Log parsing (port of `src/parse.py`)

Drain (via `drain3`) turns raw log lines into event templates. Following the paper (and Qin et
al., SANER 2025), dynamic variables (block ids, IPs, paths, numbers...) are masked by regex
**before** Drain sees them; BGL additionally uses the regexes refined in Wu et al.'s replication
package. Drain params: depth 4, similarity threshold 0.5, max children 100.

Special case: messages that are entirely wildcards get Drain similarity 0 and would each become
their own cluster — they are routed to one fixed template (`EventId 0`) instead.

Outputs (in `data/parsed/<DS>/`): `<DS>_events.csv` (one row per log line: HDFS → LineId,
BlockIds, EventId; BGL → LineId, Timestamp, Label, EventId), `<DS>_templates.csv`, and a 2000-line
sample CSV for manual inspection.

In [ ]:
# Port of src/parse.py (constants + parser factory)
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

LOG_FORMATS = {
    'HDFS': '<Date> <Time> <Pid> <Level> <Component>: <Content>',
    'BGL': '<Label> <Timestamp> <Date> <Node> <Time> <NodeRepeat> <Type> <Component> <Level> <Content>',
}

COMMON_REGEX = [
    r'blk_-?\d+',                                         # HDFS block id
    r'(/|)(\d{1,3}\.){3}\d{1,3}(:\d+)?',                  # IP (+port)
    r'0[xX][0-9a-fA-F]+',                                 # hex with 0x
    r'(?<![\w/])(/[\w.\-]+){2,}/?',                       # file paths
]
DATASET_REGEX = {
    'HDFS': [],
    'BGL': [
        # Added by the team after inspecting rare templates
        r'([0-9A-Fa-f]{2}:){5,}[0-9A-Fa-f]{2}',           # iDo chip / LP ids
        r'R\d{2}-M\d[\w:-]*',                              # node location R02-M1-N0-C:J12-U11
        # Refined regexes reported in Wu et al.'s replication package
        r'core\.\d+',
        r'(?<=r)\d{1,2}',
        r'(?<=fpr)\d{1,2}',
        r'(0x)?[0-9a-fA-F]{8}',
        r'(?<=\.\.)\d+(?!x)',
        r'\d+(?=:)',
        r'(?<=\=)\d+(?!x)',
    ],
}
NUMBER_REGEX = r'(?<![A-Za-z_<])-?\d+(\.\d+)?(?![A-Za-z_>])'  # standalone numbers
DRAIN_PARAMS = {'HDFS': dict(depth=4, sim_th=0.5), 'BGL': dict(depth=4, sim_th=0.5)}
BLOCK_RE = re.compile(r'blk_-?\d+')


def header_regex(log_format):
    parts = re.split(r'(<[^<>]+>)', log_format)
    headers, regex = [], ''
    for p in parts:
        if p.startswith('<') and p.endswith('>'):
            name = p.strip('<>')
            headers.append(name)
            regex += f'(?P<{name}>.*?)' if name != 'Content' else f'(?P<{name}>.*)'
        else:
            regex += r'\s+'.join(re.escape(s) for s in p.split(' '))
    return headers, re.compile('^' + regex + '$')


def build_masker(dataset):
    patterns = [re.compile(p) for p in COMMON_REGEX + DATASET_REGEX[dataset] + [NUMBER_REGEX]]

    def mask(text):
        for p in patterns:
            text = p.sub('<*>', text)
        # Merge adjacent wildcards; never insert whitespace next to punctuation
        # ("<*>:" must stay one token): that changes Drain's token count.
        text = re.sub(r'(<\*>)+', '<*>', text)
        text = re.sub(r'(?<!\S)<\*>(\s+<\*>)+(?!\S)', '<*>', text)
        return text.strip()
    return mask


WILDCARD_ONLY = re.compile(r'(<\*>\s*)*')
WILDCARD_EVENT_ID = 0   # drain3 gives similarity 0 to all-wildcard messages


def make_miner(ds):
    cfg = TemplateMinerConfig()
    cfg.drain_depth = DRAIN_PARAMS[ds]['depth']
    cfg.drain_sim_th = DRAIN_PARAMS[ds]['sim_th']
    cfg.drain_max_children = 100
    cfg.masking_instructions = []          # masking is done by our own preprocessing
    cfg.profiling_enabled = False
    return TemplateMiner(config=cfg)


def make_content_parser(ds, preprocess=True):
    # Returns parse(content) -> EventId, and a function giving the final {EventId: (template, size)}
    miner = make_miner(ds)
    mask = build_masker(ds) if preprocess else (lambda s: s)
    n_wild = [0]

    def parse(content):
        masked = mask(content)
        if WILDCARD_ONLY.fullmatch(masked):
            n_wild[0] += 1
            return WILDCARD_EVENT_ID
        return miner.add_log_message(masked)['cluster_id']

    def templates():
        t = {c.cluster_id: (c.get_template(), c.size) for c in miner.drain.clusters}
        if n_wild[0]:
            t[WILDCARD_EVENT_ID] = ('<*>', n_wild[0])
        return t
    return parse, templates

In [ ]:
# Port of src/parse.py main() — HDFS first run takes ~20-40 min; cached to Drive afterwards
import csv

def run_parse(ds, limit=None):
    in_path = os.path.join(REPO, 'data', 'raw', ds, ds + '.log')
    out_dir = os.path.join(REPO, 'data', 'parsed', ds)
    os.makedirs(out_dir, exist_ok=True)

    parse, get_templates = make_content_parser(ds)
    _, line_re = header_regex(LOG_FORMATS[ds])

    events_path = os.path.join(out_dir, ds + '_events.csv')
    sample_rows = []
    n, skipped, t0 = 0, 0, time.time()
    with open(in_path, encoding='utf-8', errors='replace') as fin, \
         open(events_path, 'w', newline='', encoding='utf-8') as fout:
        w = csv.writer(fout)
        w.writerow(['LineId', 'BlockIds', 'EventId'] if ds == 'HDFS'
                   else ['LineId', 'Timestamp', 'Label', 'EventId'])
        for line in fin:
            n += 1
            if limit and n > limit:
                n -= 1
                break
            line = line.rstrip('\r\n')
            m = line_re.match(line) or line_re.match(line + ' ')   # some BGL lines: empty Content
            if not m:
                skipped += 1
                continue
            content = m.group('Content') or '<EMPTY>'
            eid = parse(content)
            if ds == 'HDFS':
                w.writerow([n, ';'.join(sorted(set(BLOCK_RE.findall(content)))), eid])
            else:
                w.writerow([n, m.group('Timestamp'), 0 if m.group('Label') == '-' else 1, eid])
            if len(sample_rows) < 2000:
                sample_rows.append((n, content, eid))
            if n % 500_000 == 0:
                print(f'  {n:,} lines, {n / (time.time() - t0):,.0f} lines/s', flush=True)

    templates = get_templates()
    with open(os.path.join(out_dir, ds + '_templates.csv'), 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['EventId', 'EventTemplate', 'Occurrences'])
        for eid, (tpl, size) in sorted(templates.items(), key=lambda x: -x[1][1]):
            w.writerow([eid, tpl, size])
    with open(os.path.join(out_dir, ds + '_sample.csv'), 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['LineId', 'Content', 'EventId', 'EventTemplate'])
        for lid, content, eid in sample_rows:
            w.writerow([lid, content, eid, templates[eid][0]])
    print(f'Done {ds}: {n:,} lines ({skipped} unmatched), {len(templates)} templates, '
          f'{time.time() - t0:.0f}s')

    cdir = os.path.join(CACHE, 'parsed', ds)   # cache to Drive for next sessions
    os.makedirs(cdir, exist_ok=True)
    for f in glob.glob(os.path.join(out_dir, ds + '_*.csv')):
        shutil.copy2(f, cdir)

for ds in ('HDFS', 'BGL'):
    if os.path.exists(os.path.join(REPO, 'data', 'parsed', ds, ds + '_events.csv')):
        print(f'[skip] {ds} already parsed')
    else:
        run_parse(ds)

In [ ]:
import numpy as np
import pandas as pd

for ds in ('HDFS', 'BGL'):
    t = pd.read_csv(os.path.join(REPO, 'data', 'parsed', ds, ds + '_templates.csv'),
                    keep_default_na=False)
    print(f'{ds}: {len(t)} templates (team reference: HDFS ~45, BGL ~370)')
    print(t.head(8).to_string(index=False), '\n')

## Step 2a — Log partition (port of `src/partition.py`)

- **HDFS**: one session per `block_id`; a line mentioning several blocks belongs to all of them;
  label from Loghub `anomaly_label.csv`. Expected: **575,061 sessions, 2.9% anomalous**.
- **BGL**: fixed 6-hour windows on the epoch Timestamp column; a window is anomalous if it
  contains at least one alert line. Expected: **718 sessions**.

Events inside a session keep the original log order (LineId).

In [ ]:
BGL_WINDOW_SECONDS = 6 * 3600

def group_sequences(keys, values):
    # Group values by key, keeping the input order inside each group
    codes, uniques = pd.factorize(keys, sort=False)
    order = np.argsort(codes, kind='stable')
    starts = np.flatnonzero(np.diff(codes[order], prepend=-1))
    return uniques, np.split(values[order].astype(np.int32), starts[1:])


def sessions_hdfs():
    e = pd.read_csv(os.path.join(REPO, 'data/parsed/HDFS/HDFS_events.csv'), dtype={'BlockIds': str})
    e = e.dropna(subset=['BlockIds'])
    e = e.assign(session_id=e['BlockIds'].str.split(';')).explode('session_id')
    e = e.sort_values('LineId', kind='stable')
    ids, seqs = group_sequences(e['session_id'].to_numpy(), e['EventId'].to_numpy())
    lab = pd.read_csv(os.path.join(REPO, 'data/raw/HDFS/preprocessed/anomaly_label.csv'))
    lab = lab.set_index('BlockId')['Label'].eq('Anomaly').astype(np.int8)
    df = pd.DataFrame({'session_id': ids, 'events': seqs})
    df['label'] = df['session_id'].map(lab)
    assert df['label'].notna().all(), 'some blocks have no label'
    return df


def sessions_bgl():
    e = pd.read_csv(os.path.join(REPO, 'data/parsed/BGL/BGL_events.csv'))
    e = e.sort_values('LineId', kind='stable')
    t0 = e['Timestamp'].min()
    win = ((e['Timestamp'] - t0) // BGL_WINDOW_SECONDS).to_numpy(np.int64)
    ids, seqs = group_sequences(win, e['EventId'].to_numpy())
    labels = e.groupby(win)['Label'].max().astype(np.int8)
    df = pd.DataFrame({'session_id': ids, 'events': seqs})
    df['label'] = df['session_id'].map(labels)
    df['start_time'] = t0 + df['session_id'] * BGL_WINDOW_SECONDS
    return df.sort_values('session_id', ignore_index=True)


def build_sessions(ds):
    out = os.path.join(REPO, 'data/sessions', ds + '_sessions.pkl')
    cachef = os.path.join(CACHE, 'sessions', ds + '_sessions.pkl')
    if os.path.exists(out):
        df = pd.read_pickle(out)
    elif os.path.exists(cachef):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        shutil.copy2(cachef, out)
        df = pd.read_pickle(out)
    else:
        df = sessions_hdfs() if ds == 'HDFS' else sessions_bgl()
        df['n_events'] = df['events'].map(len)
        os.makedirs(os.path.dirname(out), exist_ok=True)
        df.to_pickle(out)
        os.makedirs(os.path.dirname(cachef), exist_ok=True)
        shutil.copy2(out, cachef)
    n, k = len(df), int(df['label'].sum())
    print(f'{ds}: {n:,} sessions, {k:,} anomalous ({df["label"].mean():.2%}) | '
          f'events/session: mean {df["n_events"].mean():.1f}, median {df["n_events"].median():.0f}, '
          f'max {df["n_events"].max():,}')
    return df

SESSIONS = {ds: build_sessions(ds) for ds in ('HDFS', 'BGL')}
# paper: HDFS 575,061 sessions (2.9% anomalous) | BGL 718 sessions

## Step 2b — Train/test split (port of `src/split.py`)

Sessions are shuffled then split (HDFS 70/30, BGL 80/20). Deviation from the paper (kept from
the team's scripts): the split is **stratified** on the label so train and test keep the same
anomaly rate — it matters on BGL, which has only ~143 test sessions. Seed 0 = main replication
split; the extension resamples with seeds 1–9.

In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_RATIO = {'HDFS': 0.7, 'BGL': 0.8}

def make_split(labels, train_ratio, seed, stratify=True):
    idx = np.arange(len(labels))
    train_idx, test_idx = train_test_split(idx, train_size=train_ratio, shuffle=True,
                                           random_state=seed, stratify=labels if stratify else None)
    return np.sort(train_idx), np.sort(test_idx)

SPLITS = {}
for ds in ('HDFS', 'BGL'):
    y = SESSIONS[ds]['label'].to_numpy()
    tr, te = make_split(y, TRAIN_RATIO[ds], SEED)
    SPLITS[ds] = (tr, te)
    print(f'{ds} seed {SEED}: train {len(tr):,} ({y[tr].sum():,} anomalous) | '
          f'test {len(te):,} ({y[te].sum():,} anomalous)')
# paper: HDFS 402,542 / 172,519 | BGL 575 / 143

## Step 3 — Representations (ports of `src/mcv.py` and `src/tfidf_text.py`)

**MCV** (for RF): raw template counts per session. The vocabulary is built from TRAIN sessions
only; events never seen in train fall into one extra `UNK` column (no test leakage into the
feature space).

**TF-IDF (Text)** (for LSTM, as in Wu et al.'s replication package `TT_TFIDF_generator.py`):
fit `TfidfVectorizer` on the TRAIN sessions (one document = the template texts of a session),
then represent each event by the TF-IDF vector of its template text. Features = template tokens
(e.g. "block", "exception"). The last embedding row is the zero padding vector.

In [ ]:
UNK = 'UNK'

def build_vocab(train_events):
    return np.unique(np.concatenate(list(train_events)))

def count_matrix(events, vocab):
    # events: iterable of int arrays -> (n_sessions, len(vocab)+1) int32 counts; last column = UNK
    col = {e: i for i, e in enumerate(vocab)}
    unk = len(vocab)
    X = np.zeros((len(events), len(vocab) + 1), dtype=np.int32)
    for r, seq in enumerate(events):
        ids, cnt = np.unique(seq, return_counts=True)
        for e, c in zip(ids, cnt):
            X[r, col.get(e, unk)] += c
    return X

def build_mcv(sessions, train_idx, test_idx):
    ev = sessions['events'].to_numpy()
    vocab = build_vocab(ev[train_idx])
    names = np.array([f'E{e}' for e in vocab] + [UNK])
    return count_matrix(ev[train_idx], vocab), count_matrix(ev[test_idx], vocab), names


def load_templates(ds):
    t = pd.read_csv(os.path.join(REPO, 'data/parsed', ds, ds + '_templates.csv'), keep_default_na=False)
    return dict(zip(t['EventId'], t['EventTemplate']))

def event_vectors(ds, sessions, train_idx):
    # E[event_id] = TF-IDF vector of its template (last row = zero padding)
    from sklearn.feature_extraction.text import TfidfVectorizer
    templates = load_templates(ds)
    train_docs = [' '.join(templates[e] for e in seq)
                  for seq in sessions['events'].to_numpy()[train_idx]]
    vec = TfidfVectorizer(lowercase=True)
    vec.fit(train_docs)
    max_id = max(templates)
    E = np.zeros((max_id + 2, len(vec.vocabulary_)), dtype=np.float32)
    ids = sorted(templates)
    E[ids] = vec.transform([templates[i] for i in ids]).toarray()
    return E, np.array(vec.get_feature_names_out())


MCV = {}
for ds in ('HDFS', 'BGL'):
    tr, te = SPLITS[ds]
    x_train, x_test, names = build_mcv(SESSIONS[ds], tr, te)
    y = SESSIONS[ds]['label'].to_numpy()
    MCV[ds] = dict(x_train=x_train, y_train=y[tr], x_test=x_test, y_test=y[te], names=names)
    const = int((x_train.min(0) == x_train.max(0)).sum())
    print(f'{ds} MCV: x_train {x_train.shape}, x_test {x_test.shape}, '
          f'{len(names) - 1} train templates + UNK, {const} constant columns, '
          f'{int((x_test[:, -1] > 0).sum())} test sessions with UNK events')

## Step 4 — Models, metrics, feature selection

**Metrics** (port of `src/evaluation.py`): precision, recall, F1 (threshold 0.5) and AUC, all on
the test sessions.

**Feature selection** (port of `src/feature_selection.py`, fitted on TRAIN only) — step 3a of the
assignment:

1. drop columns that are constant in train (VIF undefined for them)
2. correlation analysis: hierarchical clustering on Spearman |ρ| (distance 1 − |ρ|, cut at
   |ρ| = 0.7); in each cluster keep the feature with the lowest mean |ρ| to all other features
   (same rule as AutoSpearman, Jiarpakdee et al. ICSME'18)
3. redundancy analysis: iteratively drop the feature with the highest VIF until all VIF ≤ 5

**RF** (port of `src/run_rf.py`): sklearn defaults like the paper's replication package
(`n_estimators=100`), on MCV.

**LSTM** (port of `src/run_lstm.py`): windows of TF-IDF event vectors (HDFS 30/1, BGL 50/50,
shorter sessions left-padded, tail of every session covered), frozen embedding, `LSTM(hidden=8)
→ FC(2)`, class-weighted cross-entropy, Adam lr 1e-3, batch 1024, epochs HDFS 5 / BGL 30;
a session's score is the max anomaly probability over its windows.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from scipy.stats import rankdata
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

PAPER_F1 = {('RF', 'HDFS'): 0.999, ('RF', 'BGL'): 0.891,
            ('LSTM', 'HDFS'): 0.955, ('LSTM', 'BGL'): 0.871}


def evaluate(y_true, scores, threshold=0.5):
    y_pred = (np.asarray(scores) > threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    return {'precision': round(float(p), 4), 'recall': round(float(r), 4),
            'f1': round(float(f1), 4), 'auc': round(float(roc_auc_score(y_true, scores)), 4)}


def save_result(name, result):
    out_dir = os.path.join(REPO, 'results', 'rq1_notebook')
    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, name + '.json')
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)
    cdir = os.path.join(CACHE, 'results')
    os.makedirs(cdir, exist_ok=True)
    shutil.copy2(out, cdir)
    return out


CORR_THRESHOLD = 0.7
VIF_THRESHOLD = 5.0

def vif(X):
    # VIF_j = 1 / (1 - R^2_j) = diag of the inverse correlation matrix
    corr = np.corrcoef(X, rowvar=False)
    return np.diag(np.linalg.pinv(corr))


def select_features(X_train, names):
    # Returns (kept column indices, report dict)
    X = np.asarray(X_train, dtype=np.float64)
    cols = np.arange(X.shape[1])

    nonconst = X.std(axis=0) > 0
    dropped_const = names[cols[~nonconst]].tolist()
    cols = cols[nonconst]

    dropped_corr = []
    if len(cols) > 1:
        ranks = np.apply_along_axis(rankdata, 0, X[:, cols])
        rho = np.abs(np.nan_to_num(np.corrcoef(ranks, rowvar=False)))
        np.fill_diagonal(rho, 1.0)
        dist = squareform(1 - rho, checks=False).clip(min=0)
        clusters = fcluster(linkage(dist, method='average'), t=1 - CORR_THRESHOLD, criterion='distance')
        mean_rho = (rho.sum(axis=1) - 1) / max(len(cols) - 1, 1)
        keep = []
        for c in np.unique(clusters):
            members = np.flatnonzero(clusters == c)
            best = members[np.argmin(mean_rho[members])]
            keep.append(best)
            dropped_corr += names[cols[np.setdiff1d(members, best)]].tolist()
        cols = cols[np.sort(keep)]

    dropped_vif = []
    while len(cols) > 1:
        v = vif(X[:, cols])
        worst = int(np.argmax(v))
        if v[worst] <= VIF_THRESHOLD:
            break
        dropped_vif.append((names[cols[worst]], round(float(v[worst]), 1)))
        cols = np.delete(cols, worst)

    report = {'n_in': len(names), 'n_out': len(cols), 'dropped_constant': dropped_const,
              'dropped_correlation': dropped_corr, 'dropped_vif': dropped_vif,
              'kept': names[cols].tolist()}
    return cols, report

In [ ]:
# Port of src/run_rf.py — Random Forest on MCV, with and without feature selection
from sklearn.ensemble import RandomForestClassifier

def run_rf(ds, seed=SEED, select='none'):
    d = MCV[ds]
    x_train, y_train, x_test, y_test, names = d['x_train'], d['y_train'], d['x_test'], d['y_test'], d['names']
    report = None
    if select == 'vif':
        cols, report = select_features(x_train, names)
        x_train, x_test = x_train[:, cols], x_test[:, cols]
    t0 = time.time()
    clf = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)  # sklearn defaults
    clf.fit(x_train, y_train)
    scores = clf.predict_proba(x_test)[:, 1]
    res = {'model': 'RF', 'representation': 'MCV', 'dataset': ds, 'seed': seed, 'select': select,
           'n_features': int(x_train.shape[1]), 'train_time_s': round(time.time() - t0, 1),
           **evaluate(y_test, scores), 'selection_report': report}
    save_result(f'RF_MCV_{ds}_{select}_seed{seed}', res)
    print(f'RF {ds} select={select}: features={res["n_features"]} P={res["precision"]} '
          f'R={res["recall"]} F1={res["f1"]} AUC={res["auc"]} ({res["train_time_s"]}s)')
    return res

RF_RES = {(ds, sel): run_rf(ds, SEED, sel) for ds in ('HDFS', 'BGL') for sel in ('none', 'vif')}

In [ ]:
# Port of src/run_lstm.py — LSTM on TF-IDF (Text) event vectors (GPU if available)
import torch
from torch import nn

WINDOW = {'HDFS': (30, 1), 'BGL': (50, 50)}
EPOCHS = {'HDFS': 5, 'BGL': 30}
BATCH = 1024
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def make_windows(events, size, stride, pad_id):
    # -> windows (n, size) of event ids (shorter sessions left-padded), session index per window
    wins, owner = [], []
    for s, seq in enumerate(events):
        if len(seq) < size:
            seq = np.concatenate([np.full(size - len(seq), pad_id, np.int32), seq])
        starts = range(0, len(seq) - size + 1, stride)
        last = len(seq) - size
        if starts[-1] != last:          # make sure the tail of the session is covered
            starts = list(starts) + [last]
        for st in starts:
            wins.append(seq[st:st + size])
        owner += [s] * len(starts)
    return np.stack(wins).astype(np.int64), np.array(owner)


class LSTMClassifier(nn.Module):
    def __init__(self, E):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.from_numpy(E), freeze=True)
        self.lstm = nn.LSTM(E.shape[1], 8, batch_first=True)
        self.fc = nn.Linear(8, 2)

    def forward(self, x):
        out, _ = self.lstm(self.emb(x))
        return self.fc(out[:, -1])


def session_scores(model, X, owner, n_sessions):
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), 8192):
            xb = torch.from_numpy(X[i:i + 8192]).to(DEVICE)
            probs.append(torch.softmax(model(xb), 1)[:, 1].cpu().numpy())
    scores = np.zeros(n_sessions)
    np.maximum.at(scores, owner, np.concatenate(probs))   # session score = max over its windows
    return scores


def run_lstm(ds, seed=SEED, select='none', epochs=None):
    epochs = epochs or EPOCHS[ds]
    torch.manual_seed(seed)
    np.random.seed(seed)

    sessions = SESSIONS[ds]
    tr, te = SPLITS[ds]
    E, names = event_vectors(ds, sessions, tr)
    pad_id = len(E) - 1

    report = None
    if select == 'vif':
        # rows = sequence-level (mean) vectors of the train sessions, columns = tokens
        agg = np.stack([E[seq].mean(0) for seq in sessions['events'].to_numpy()[tr]])
        cols, report = select_features(agg, names)
        E = np.ascontiguousarray(E[:, cols])

    size, stride = WINDOW[ds]
    ev, y = sessions['events'].to_numpy(), sessions['label'].to_numpy()
    Xtr, own_tr = make_windows(ev[tr], size, stride, pad_id)
    Xte, own_te = make_windows(ev[te], size, stride, pad_id)
    ytr = y[tr][own_tr].astype(np.int64)
    print(f'{ds}: {E.shape[1]} token features, {len(Xtr):,} train windows ({ytr.mean():.1%} anomalous), '
          f'{len(Xte):,} test windows', flush=True)

    model = LSTMClassifier(E).to(DEVICE)
    w = torch.tensor(len(ytr) / (2 * np.bincount(ytr, minlength=2).clip(min=1)),
                     dtype=torch.float32).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm, total = np.random.permutation(len(Xtr)), 0.0
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            xb = torch.from_numpy(Xtr[b]).to(DEVICE)
            yb = torch.from_numpy(ytr[b]).to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        print(f'  epoch {ep + 1}/{epochs} loss {total / len(perm):.4f} ({time.time() - t0:.0f}s)', flush=True)

    scores = session_scores(model, Xte, own_te, len(te))
    res = {'model': 'LSTM', 'representation': 'TF-IDF (Text)', 'dataset': ds, 'seed': seed,
           'select': select, 'n_features': int(E.shape[1]), 'epochs': epochs,
           'train_time_s': round(time.time() - t0, 1), **evaluate(y[te], scores),
           'selection_report': report}
    save_result(f'LSTM_TFIDF_{ds}_{select}_seed{seed}', res)
    print(f'LSTM {ds} select={select}: features={res["n_features"]} P={res["precision"]} '
          f'R={res["recall"]} F1={res["f1"]} AUC={res["auc"]} ({res["train_time_s"]}s)')
    return res

print('device:', DEVICE)
LSTM_RES = {(ds, sel): run_lstm(ds, SEED, sel) for ds in ('HDFS', 'BGL') for sel in ('none', 'vif')}

In [ ]:
# Summary: our results vs the paper (port of the seed-0 part of src/summarize_rq1.py)
rows = [{k: res[k] for k in ['model', 'representation', 'dataset', 'select', 'n_features',
                             'precision', 'recall', 'f1', 'auc', 'train_time_s']}
        for res in list(RF_RES.values()) + list(LSTM_RES.values())]
df = pd.DataFrame(rows)
df['paper_f1'] = [PAPER_F1[(m, d)] for m, d in zip(df.model, df.dataset)]
df['f1_gap_vs_paper'] = (df.f1 - df.paper_f1).round(4)
df

## Notes

- Numbers should land close to the team reference table at the top (small drift is expected:
  drain3 template ids and LSTM training are only seeded, not bit-deterministic on GPU).
- `results/rq1_notebook/*.json` holds one JSON per run (same schema as the team's
  `results/rq1/*.json`), also mirrored to the Drive cache.
- To reproduce the 10-seed statistics of `results/rq1_summary.csv`, re-run the model cells with
  `SEED = 1..9` (splits/features must be rebuilt per seed — the scripts in `src/` already loop
  over seeds; the notebook keeps seed 0 for clarity).
- Differences vs the scripts: paths/cache adapted to Colab, LSTM optionally on GPU. Algorithms,
  hyper-parameters, window configs and seeds are identical.